<a href="https://colab.research.google.com/github/awaisfarooqchaudhry/IB9AU-GenerativeAI-2026/blob/main/Task14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 14 — Page-Wise Visual RAG for AstraZeneca FY & Q4 2025 Earnings (Google Colab)

This notebook is a **complete beginner-friendly Colab notebook** for **Required Task 14**.

It is adapted from your uploaded reference notebook **`RAG_5_Multimodal_Chunking_OpenSourced.ipynb`**, but rewritten as a **standalone Colab workflow** that is easier to run and debug.

## What Task 14 asks you to do
Build a **Page-Wise Visual RAG (Retrieval-Augmented Generation)** system to analyze **AstraZeneca's FY and Q4 2025 earnings report**.

The assignment specifically wants you to:
- retrieve the **most relevant page** for a query
- render that page as an **image**
- use a **Vision-Language Model (VLM)** to answer the query visually
- answer the 3 required AstraZeneca queries
- create an **audit-style query** with page-aware verification

## Required setup from the assignment
- **Embeddings:** `sentence-transformers/all-MiniLM-L6-v2` (local, CPU)
- **VLM:** `Qwen/Qwen2.5-VL-3B-Instruct` (local, T4 GPU)
- **PDF:** `AstraZeneca-Q4-2025-earnings.pdf`
- **Runtime:** Google Colab with **T4 GPU**

## What this notebook does
1. uploads the earnings PDF
2. extracts text **page by page**
3. creates **page embeddings** using `all-MiniLM-L6-v2`
4. retrieves the most relevant page for a query
5. renders that page as a high-resolution image
6. asks `Qwen2.5-VL-3B-Instruct` to answer based on the page image
7. runs all required Task 14 queries
8. saves results and page images for submission

## Colab recommendation
Use **Runtime → Change runtime type → T4 GPU** before loading the VLM.


## Why this notebook is slightly simpler than the lab notebook

Your uploaded lab notebook uses a LlamaIndex-based page-wise workflow.

This version keeps the **same core idea** but makes the retrieval layer simpler and more stable for Colab:
- **manual page-wise text extraction** with PyMuPDF
- **manual page embeddings** with `sentence-transformers`
- **manual cosine-similarity retrieval**
- **visual answer generation** with Qwen2.5-VL on the rendered page image

This still fully satisfies the assignment logic:
- retrieve the correct page first
- then read the page visually like an analyst


## Step 0 — Install packages

Run this cell first.

In [ ]:
!pip install -q "pillow<12" pymupdf sentence-transformers transformers accelerate

## Step 1 — Imports and runtime checks

In [ ]:
!pip install Pillow==9.5.0
import os
import json
from pathlib import Path

import fitz  # PyMuPDF
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from IPython.display import display, Markdown

import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU detected. The VLM will be very slow on CPU. Switch to a T4 GPU in Colab if possible.")

OUTPUT_DIR = Path("task14_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## Step 2 — Upload `AstraZeneca-Q4-2025-earnings.pdf`

You said you will upload this file yourself, so this notebook expects you to do that here.


In [ ]:
from google.colab import files

uploaded = files.upload()

if len(uploaded) == 0:
    raise ValueError("No file uploaded. Please upload AstraZeneca-Q4-2025-earnings.pdf")

uploaded_name = list(uploaded.keys())[0]
PDF_PATH = f"/content/{uploaded_name}"

print("✅ Uploaded PDF:", PDF_PATH)

## Step 3 — Inspect the PDF and extract page text

We extract:
- page number
- page text
- character count

This text is used **only for retrieval**.
The final answer still comes from the **page image** via the VLM.


In [ ]:
def extract_pages_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    rows = []

    for i, page in enumerate(doc):
        text = page.get_text("text")
        rows.append({
            "page_number": i + 1,
            "text": text if text is not None else "",
            "char_count": len(text) if text is not None else 0
        })

    return pd.DataFrame(rows), doc.page_count

df_pages, page_count = extract_pages_from_pdf(PDF_PATH)

print("Number of pages:", page_count)
display(df_pages.head())

## Step 4 — Quick preview of the extracted page text

This is useful for debugging retrieval later.

In [ ]:
preview_page = 1
row = df_pages[df_pages["page_number"] == preview_page].iloc[0]

print(f"===== PAGE {preview_page} TEXT PREVIEW =====")
print(row["text"][:4000])

## Step 5 — Load the embedding model on CPU

The assignment requires:
- `sentence-transformers/all-MiniLM-L6-v2`

This model is used only for **page retrieval**, not for answering.


In [ ]:
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")
print("✅ Embedding model loaded:", EMBED_MODEL_NAME)

## Step 6 — Create page embeddings

Each page becomes one vector.
Later, we compare the query vector to all page vectors and retrieve the most similar page.


In [ ]:
page_texts = df_pages["text"].fillna("").tolist()

page_embeddings = embed_model.encode(
    page_texts,
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Page embedding matrix shape:", page_embeddings.shape)

## Step 7 — Load the VLM on the GPU

The assignment requires:
- `Qwen/Qwen2.5-VL-3B-Instruct`

This model reads the **rendered page image** and answers the query visually.


In [ ]:
VLM_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

vlm = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_MODEL_ID,
    torch_dtype=torch_dtype,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)

print("✅ VLM loaded:", VLM_MODEL_ID)

## Step 8 — Helper functions

These functions do the main Visual RAG work:
- retrieve top pages
- render a page as an image
- ask the VLM to answer from that page


In [ ]:
def cosine_search(query, top_k=3):
    query_emb = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    scores = page_embeddings @ query_emb
    top_idx = np.argsort(scores)[::-1][:top_k]

    results = df_pages.iloc[top_idx].copy()
    results = results.assign(similarity_score=scores[top_idx])
    results = results.sort_values(by="similarity_score", ascending=False).reset_index(drop=True)
    return results


def render_pdf_page(pdf_path, page_number, zoom=2.2, max_width=1600):
    doc = fitz.open(pdf_path)
    page = doc[page_number - 1]

    matrix = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=matrix, alpha=False)

    image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

    if image.width > max_width:
        new_height = int(image.height * (max_width / image.width))
        image = image.resize((max_width, new_height))

    return image


def show_page_image(page_image, title=None, figsize=(12, 14)):
    plt.figure(figsize=figsize)
    plt.imshow(page_image)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()


def ask_vlm_from_image(page_image, user_query, page_number, max_new_tokens=280):
    prompt = f"""
You are a careful financial analyst reading a single page from AstraZeneca's FY and Q4 2025 earnings report.

Instructions:
- Answer ONLY from the visible page image.
- If the page does not contain enough information, say so clearly.
- Preserve exact figures, dates, and medicine names where possible.
- Mention the page number in your answer.
- If you are reading a table, keep row/column meanings aligned carefully.
- Do not guess.

User query:
{user_query}
""".strip()

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": page_image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text_input = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text_input],
        images=[page_image],
        padding=True,
        return_tensors="pt"
    )

    inputs = {k: v.to(vlm.device) for k, v in inputs.items()}

    with torch.no_grad():
        generated_ids = vlm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_ids_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs["input_ids"], generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )[0]

    return output_text


def query_visual_rag(query, top_k=3, chosen_rank=1, zoom=2.2, max_new_tokens=280, display_page=True):
    retrieval = cosine_search(query, top_k=top_k)

    if chosen_rank < 1 or chosen_rank > len(retrieval):
        raise ValueError("chosen_rank must be between 1 and top_k")

    chosen_row = retrieval.iloc[chosen_rank - 1]
    chosen_page = int(chosen_row["page_number"])

    page_image = render_pdf_page(PDF_PATH, chosen_page, zoom=zoom)

    if display_page:
        show_page_image(page_image, title=f"Retrieved Page {chosen_page}")

    answer = ask_vlm_from_image(
        page_image=page_image,
        user_query=query,
        page_number=chosen_page,
        max_new_tokens=max_new_tokens
    )

    return {
        "query": query,
        "retrieval": retrieval,
        "chosen_page": chosen_page,
        "page_image": page_image,
        "answer": answer
    }

## Step 9 — Sanity check the retrieval system

Try a simple finance question and see which pages come back first.

In [ ]:
test_query = "total revenue FY 2025 AstraZeneca"

test_retrieval = cosine_search(test_query, top_k=5)
display(test_retrieval[["page_number", "similarity_score", "char_count"]])

## Step 10 — Run Task 1

**Task 1 query:**
> What were AstraZeneca's total Product Sales and Alliance Revenue for FY 2025, and how did each change compared to FY 2024?


In [ ]:
task1_query = "What were AstraZeneca's total Product Sales and Alliance Revenue for FY 2025, and how did each change compared to FY 2024?"

task1_result = query_visual_rag(task1_query, top_k=3, chosen_rank=1, zoom=2.4, max_new_tokens=300)

display(task1_result["retrieval"][["page_number", "similarity_score"]])
print("\nRetrieved page:", task1_result["chosen_page"])
print("\nAnswer:")
print(task1_result["answer"])

## Step 11 — Run Task 2

**Task 2 query:**
> Which geographic region had the highest Total Revenue growth in FY 2025, and what was the growth rate at constant exchange rates?


In [ ]:
task2_query = "Which geographic region had the highest Total Revenue growth in FY 2025, and what was the growth rate at constant exchange rates?"

task2_result = query_visual_rag(task2_query, top_k=3, chosen_rank=1, zoom=2.4, max_new_tokens=300)

display(task2_result["retrieval"][["page_number", "similarity_score"]])
print("\nRetrieved page:", task2_result["chosen_page"])
print("\nAnswer:")
print(task2_result["answer"])

## Step 12 — Run Task 3

**Task 3 query:**
> Which medicines received regulatory approvals in the US between November 2025 and February 2026, and for what indications?


In [ ]:
task3_query = "Which medicines received regulatory approvals in the US between November 2025 and February 2026, and for what indications?"

task3_result = query_visual_rag(task3_query, top_k=4, chosen_rank=1, zoom=2.4, max_new_tokens=360)

display(task3_result["retrieval"][["page_number", "similarity_score"]])
print("\nRetrieved page:", task3_result["chosen_page"])
print("\nAnswer:")
print(task3_result["answer"])

## Step 13 — Task 4: Audit Mode Query

The assignment asks you to design your own audit-style prompt that:
- demands precise numerical figures
- explicitly asks for the page or table
- targets a different section from Tasks 1–3

This notebook uses a **cash-flow style** audit prompt by default.
You can edit it if you want.


In [ ]:
audit_query = """
From the cash flow section, identify AstraZeneca's FY 2025:
1. net cash inflow from operating activities, and
2. purchases of property, plant and equipment.

Requirements:
- give exact figures
- explicitly cite the page number
- mention the relevant table if it is visible
- do not estimate
""".strip()

audit_result = query_visual_rag(audit_query, top_k=4, chosen_rank=1, zoom=2.5, max_new_tokens=320)

display(audit_result["retrieval"][["page_number", "similarity_score"]])
print("\nRetrieved page:", audit_result["chosen_page"])
print("\nAnswer:")
print(audit_result["answer"])

## Step 14 — Verify two specific figures from the retrieved audit page

The assignment asks you to verify at least two specific figures against the source document.

This cell re-reads the **same retrieved page image** with two focused verification prompts.


In [ ]:
audit_page_image = audit_result["page_image"]
audit_page_number = audit_result["chosen_page"]

verify_prompt_1 = """
From this page only, what is the exact FY 2025 figure for net cash inflow from operating activities?
Return only a short answer with:
- the figure
- the page number
- the table name if visible
""".strip()

verify_prompt_2 = """
From this page only, what is the exact FY 2025 figure for purchases of property, plant and equipment?
Return only a short answer with:
- the figure
- the page number
- the table name if visible
""".strip()

verification_1 = ask_vlm_from_image(audit_page_image, verify_prompt_1, audit_page_number, max_new_tokens=160)
verification_2 = ask_vlm_from_image(audit_page_image, verify_prompt_2, audit_page_number, max_new_tokens=160)

print("Verification 1:")
print(verification_1)
print("\nVerification 2:")
print(verification_2)

## Step 15 — Optional: if a task retrieves the wrong page, try the 2nd-ranked page

Sometimes the top page is close but not perfect.
This helper cell lets you retry using the 2nd or 3rd retrieved page.


In [ ]:
# Example:
# task1_retry = query_visual_rag(task1_query, top_k=3, chosen_rank=2, zoom=2.4, max_new_tokens=300)
# display(task1_retry["retrieval"][["page_number", "similarity_score"]])
# print(task1_retry["answer"])

## Step 16 — Build a compact results table

This gives you one clean table of:
- query
- retrieved page
- answer


In [ ]:
results_rows = [
    {
        "task": "Task 1",
        "query": task1_query,
        "retrieved_page": task1_result["chosen_page"],
        "answer": task1_result["answer"]
    },
    {
        "task": "Task 2",
        "query": task2_query,
        "retrieved_page": task2_result["chosen_page"],
        "answer": task2_result["answer"]
    },
    {
        "task": "Task 3",
        "query": task3_query,
        "retrieved_page": task3_result["chosen_page"],
        "answer": task3_result["answer"]
    },
    {
        "task": "Task 4 Audit",
        "query": audit_query,
        "retrieved_page": audit_result["chosen_page"],
        "answer": audit_result["answer"]
    },
]

results_df = pd.DataFrame(results_rows)
display(results_df)

## Step 17 — Save the retrieved page images and answers

This creates files you can download or include in your submission.

In [ ]:
(OUTPUT_DIR / "pages").mkdir(exist_ok=True)

task1_result["page_image"].save(OUTPUT_DIR / "pages" / f"task1_page_{task1_result['chosen_page']}.png")
task2_result["page_image"].save(OUTPUT_DIR / "pages" / f"task2_page_{task2_result['chosen_page']}.png")
task3_result["page_image"].save(OUTPUT_DIR / "pages" / f"task3_page_{task3_result['chosen_page']}.png")
audit_result["page_image"].save(OUTPUT_DIR / "pages" / f"audit_page_{audit_result['chosen_page']}.png")

results_df.to_csv(OUTPUT_DIR / "task14_results.csv", index=False)

summary = {
    "task1_retrieved_page": int(task1_result["chosen_page"]),
    "task2_retrieved_page": int(task2_result["chosen_page"]),
    "task3_retrieved_page": int(task3_result["chosen_page"]),
    "audit_retrieved_page": int(audit_result["chosen_page"]),
    "verification_1": verification_1,
    "verification_2": verification_2,
}

with open(OUTPUT_DIR / "task14_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("✅ Saved outputs to:", OUTPUT_DIR)

## Step 18 — Download the main outputs

In [ ]:
from google.colab import files

files.download(str(OUTPUT_DIR / "task14_results.csv"))
files.download(str(OUTPUT_DIR / "task14_summary.json"))

## Step 19 — Optional: zip the entire outputs folder

In [ ]:
import shutil

archive_path = shutil.make_archive("task14_outputs_archive", "zip", OUTPUT_DIR)
print("✅ Created archive:", archive_path)

from google.colab import files
files.download(archive_path)

## Troubleshooting

### If the VLM is too slow
Make sure you are using:
- **Runtime → Change runtime type → T4 GPU**

### If Colab runs out of memory
Try:
- reducing `zoom` from `2.5` to `2.0`
- reducing `max_new_tokens`
- running one task at a time

### If retrieval finds the wrong page
Use:
```python
query_visual_rag(query, top_k=3, chosen_rank=2)
```
or:
```python
query_visual_rag(query, top_k=4, chosen_rank=3)
```

### If the page image is too small for a table
Increase:
```python
zoom=2.6
```
or:
```python
zoom=3.0
```

### If Task 4 retrieves the wrong section
Rewrite the audit query more explicitly, for example by naming:
- cash flow
- Table 12
- reconciliation
- currency sensitivity
